Doing basic check on what needs to be done, as well as counting values

In [0]:
df = spark.sql("SELECT * FROM marathos_list.bronze.raw_marathos_list")

display(df)

In [0]:
print(f"Number of total rows {df.count()}")
print(f"Number of columns {len(df.columns)}")


checking validty of performance and event length

In [0]:
from pyspark.sql.functions import col, regexp_replace, split, round as sparkround

#effecively copied the ai, as it had the only solution
df_time = df.filter(col("Athlete performance").contains("h") & (~df["Athlete performance"].contains("d")))

# Convert time string to decimal hours
df_time = df_time.withColumn(
    "Athlete performance",
    split(regexp_replace(col("Athlete performance"), " h$", ""), ":")
).withColumn(
    "Athlete performance",
    sparkround(col("Athlete performance")[0].cast("double") + 
    col("Athlete performance")[1].cast("double") / 60 + 
    col("Athlete performance")[2].cast("double") / 3600, 2)
)

df_time.display()

seeing how many null values are in the columns

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
null_counts = df_time.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

In [0]:
null_counts = null_counts.collect()[0].asDict()

[(column, nulls) for column, nulls in null_counts.items() if nulls >0]

In [0]:
#the ai recommened that I do this instead of coalesce, this is to remove all values with nulls
df_clean = df_time.dropna(subset=[
    "Athlete club",
    "Athlete country",
    "Athlete year of birth",
    "Athlete gender",
    "Athlete age category",
    "Athlete average speed"
])

df_clean.display()

In [0]:
df_clean.createOrReplaceTempView("df_clean")
spark.sql("""
SELECT
MIN(`Year of event`) - 70 AS min_year,
MAX(`Year of event`) - 18 AS max_year
FROM
df_clean
""").display()

In [0]:
from pyspark.sql.functions import col

# Cast athlete_average_speed to double
df_clean = df_clean.withColumn("Athlete average speed", col("Athlete average speed").cast("double"))

df_clean.display()

## DONT RUN THIS YET

In [0]:
from pyspark.sql.functions import dense_rank
from pyspark.sql.window import Window

window_spec = Window.orderBy("Event name")
df_clean = df_clean.withColumn("event_id", dense_rank().over(window_spec))

df_clean.display()

In [0]:
import re
def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

In [0]:
df_column_alias = rename_columns_to_snake_case(df_clean)

df_column_alias.limit(5).display()